In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import sys
from pathlib import Path
from torchvision.datasets.utils import download_url
import graph_print_analysis as gp_tool
# add repo root so swiss_roll_models can be imported from anywhere
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "swiss_roll_models").exists():
        sys.path.insert(0, str(p))
        break

from environment.hilbert_distance import hilbert_analysis as hda

# ============================
# 1. Define a simple MNIST model
# ============================

class MNISTNet(nn.Module):
    def __init__(self, number_of_layerss=1, hidden_dim=256):
        super().__init__()
        if number_of_layerss < 1:
            raise ValueError("number_of_layerss must be at least 1.")

        layers = []
        input_dim = 28 * 28
        for _ in range(number_of_layerss):
            layers.append(nn.Linear(input_dim, hidden_dim))
            layers.append(nn.ReLU())
            input_dim = hidden_dim
        self.hidden_layers = nn.Sequential(*layers) if layers else nn.Identity()
        self.output_layer = nn.Linear(input_dim, 10)  # We track the weights of this layer

    def forward(self, x):
        x = x.view(x.size(0), -1)   # flatten
        x = self.hidden_layers(x)
        logits = self.output_layer(x)
        return logits





In [ ]:
def train_mnist_with_hilbert(
    number_of_layerss,
    num_epochs=None,      # 👈 经典 epoch 模式
    max_steps=None,       # 👈 固定 step 数模式
    batch_size=128,
    lr=1e-2,
    device=None,
    if_regularize=True,
    if_decay=False,
    loss_type="ce",
    huber_beta=1.0,
    regularization_coeff=1e-4,
    if_regularize_all=False,
    trajectory_save_path=None,
):
    """Train MNIST classifier while tracking output-layer trajectories and optional regularization.

    Args:
        number_of_layerss: Number of hidden linear/ReLU blocks to include (>=1).
        num_epochs: Number of full epochs to run; mutually exclusive with max_steps.
        max_steps: Fixed number of training steps when set; mutually exclusive with num_epochs.
        batch_size: Mini-batch size for training.
        lr: Learning rate for the SGD optimizer.
        device: Optional device override (defaults to CUDA when available).
        if_regularize: Whether to apply weight decay regularization.
        if_decay: Legacy flag preserved for compatibility; when True applies decay to all parameters.
        loss_type: Loss choice: "ce", "huber", or "mse".
        huber_beta: Beta parameter for SmoothL1 loss when loss_type="huber".
        regularization_coeff: Weight decay coefficient when regularization is enabled.
        if_regularize_all: When True, apply regularization to all parameters; otherwise only the output layer.
        trajectory_save_path: Absolute path to save the parameter trajectory; when None, do not save.
    """
    # ------------ 基本参数检查 ------------
    if (num_epochs is None) and (max_steps is None):
        raise ValueError("必须在 num_epochs 和 max_steps 里至少指定一个。")
    if (num_epochs is not None) and (max_steps is not None):
        raise ValueError("只能二选一：要么用 num_epochs，要么用 max_steps。")
    if number_of_layerss < 1:
        raise ValueError("number_of_layerss must be at least 1.")

    output_log = ""
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    output_log += f"Using device: {device}\n"

    # ---- cuDNN 加速（对卷积网络一般有帮助）----
    if device.startswith("cuda"):
        torch.backends.cudnn.benchmark = True

    # ---- Random seed (for reproducibility) ----
    torch.manual_seed(42)

    # ---- MNIST data ----
    transform = transforms.Compose([
        transforms.ToTensor(),                  # [0, 1]
        transforms.Normalize((0.1307,), (0.3081,)),
    ])

    train_dataset = datasets.MNIST(
        root="./data",
        train=True,
        download=True,
        transform=transform,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=7,      # 多进程加载
        pin_memory=True,    # 加速 CPU→GPU 拷贝
    )

    model = MNISTNet(number_of_layerss=number_of_layerss).to(device)

    # 根据 loss_type 选择不同的 criterion
    if loss_type == "ce":
        criterion = nn.CrossEntropyLoss()
    elif loss_type == "huber":
        # Huber/SmoothL1，对 logits 和 one-hot target 做
        criterion = nn.SmoothL1Loss(beta=huber_beta)
    elif loss_type == "mse":
        criterion = nn.MSELoss()
    else:
        raise ValueError(f"Unknown loss_type: {loss_type}")


    if if_regularize:
        if if_regularize_all or if_decay:
            optimizer = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=regularization_coeff)
        else:
            optimizer = torch.optim.SGD(
                [
                    {"params": model.hidden_layers.parameters(), "weight_decay": 0},
                    {"params": model.output_layer.parameters(), "weight_decay": regularization_coeff},
                ],
                lr=lr
            )
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    # ---- 用于存储参数轨迹（最后一层 output_layer.weight 的真实向量）----
    # 不做任何 abs / eps / mask / threshold 处理，全部留到外部分析函数统一处理
    param_traj = []

    # 先记录初始权重 w_0
    with torch.no_grad():
        w0 = model.output_layer.weight.detach().cpu().reshape(-1).clone()
        param_traj.append(w0)

    # =======================
    # Training loop
    # =======================
    global_step = 0
    epoch_idx = 0

    # 训练条件：
    # - epoch 模式：epoch_idx < num_epochs
    # - step  模式：global_step < max_steps
    def should_continue():
        cond_epoch = (num_epochs is None) or (epoch_idx < num_epochs)
        cond_steps = (max_steps is None) or (global_step < max_steps)
        return cond_epoch and cond_steps

    while should_continue():
        model.train()
        # running_loss = 0.0
        # correct = 0
        # total = 0

        for batch_idx, (images, labels) in enumerate(train_loader):
            # 如果是固定 step 模式，先检查是否已经够了
            if (max_steps is not None) and (global_step >= max_steps):
                break

            # non_blocking=True 在 pin_memory=True 时可以略微提升吞吐
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()
            logits = model(images)

            if loss_type == "ce":
                # 标准分类交叉熵
                loss = criterion(logits, labels)

            else:
                # 先把 label 变成 one-hot
                # logits.shape: [B, 10]
                # labels.shape: [B]
                target_onehot = F.one_hot(labels, num_classes=10).float()

                if loss_type == "huber":
                    # 对 logits 和 one-hot target 做 Huber (SmoothL1)
                    # 这里 target_onehot 已经是 0/1，逻辑上相当于让正确类的 logit 逼近 1，其余逼近 0
                    loss = criterion(logits, target_onehot)

                elif loss_type == "mse":
                    # 纯 MSE 版本
                    loss = criterion(logits, target_onehot)
            loss.backward()
            optimizer.step()

            # # Track training accuracy (optional)
            # running_loss += loss.item() * images.size(0)
            # _, predicted = logits.max(1)
            # correct += (predicted == labels).sum().item()
            # total += labels.size(0)

            # ====== 每次参数更新之后，记录 output_layer.weight 的真实向量 ======
            with torch.no_grad():
                w_t = model.output_layer.weight.detach().cpu().reshape(-1).clone()
                param_traj.append(w_t)

            global_step += 1

            # 再次检查 step 是否超限（防止多跑）
            if (max_steps is not None) and (global_step >= max_steps):
                break

        # epoch 级别的 log 你暂时注释掉了，就保持不动
        epoch_idx += 1

        if not should_continue():
            break

    steps_run = len(param_traj) - 1  # 去掉初始 w0
    output_log += (
        f"Training finished. Recorded steps (updates) = {steps_run}, "
        f"trajectory length (including init) = {len(param_traj)}\n"
    )

    # =======================
    # 这里不再做任何 Hilbert 分析，只是简单留下最终权重
    # =======================
    # w_star = param_traj[-1]

    if trajectory_save_path is not None:
        traj_path = Path(trajectory_save_path)
        if not traj_path.is_absolute():
            raise ValueError("trajectory_save_path must be an absolute path.")
        traj_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(param_traj, traj_path)
        output_log += f"Trajectory saved to {traj_path}\n"

    # 统一交给外部 analysis(...) 去做 Hilbert / mask / threshold 等等
    return {
        "model": model,
        "param_traj": param_traj,
        "output_log": output_log,
        "batch_size": batch_size,
        "lr": lr,
        "epochs_or_steps": f"steps{max_steps}" if max_steps is not None else f"ep{num_epochs}",
    }




In [ ]:

result=train_mnist_with_hilbert(
    num_epochs=3,
    batch_size=128,
    lr=1e-2,
)

Using device: cuda
Epoch [1/3]  Loss: 0.7633  Acc: 82.10%
Epoch [2/3]  Loss: 0.3615  Acc: 89.85%
Epoch [3/3]  Loss: 0.3080  Acc: 91.19%
Training finished. Total recorded steps (including init): 1408

=== Hilbert 分析结果 (部分) ===
len(hilbert_to_final)  = 1408  # 每个 step 对 w* 的距离
len(hilbert_to_init)   = 1408   # 每个 step 对 w0 的距离
len(hilbert_between)   = 1407  # 相邻步之间的距离

前 10 步 d_H(w_t, w*):
[13.656726837158203, 13.419578552246094, 13.898550033569336, 13.527496337890625, 12.841720581054688, 13.374065399169922, 12.373943328857422, 14.936422348022461, 13.295469284057617, 13.550796508789062]

前 10 步 d_H(w_t, w_0):
[0.0, 2.9013776779174805, 4.150603294372559, 5.575801849365234, 4.823336601257324, 6.833193778991699, 5.641814231872559, 8.132007598876953, 7.649444580078125, 7.533379077911377]

前 10 步 d_H(w_t, w_{t-1}):
[2.9013776779174805, 2.0438406467437744, 3.8183467388153076, 1.483393669128418, 3.898033618927002, 3.74942684173584, 4.775698661804199, 4.3495988845825195, 4.156457424163818, 4.757

In [ ]:
hilbert_to_final=result["hilbert_to_final"]
hilbert_to_init=result["hilbert_to_init"]
hilbert_between=result["hilbert_between"]


In [ ]:
print("\n=== Hilbert 分析结果 (部分) ===")
print(f"len(hilbert_to_final)  = {len(hilbert_to_final)}  # 每个 step 对 w* 的距离")
print(f"len(hilbert_to_init)   = {len(hilbert_to_init)}   # 每个 step 对 w0 的距离")
print(f"len(hilbert_between)   = {len(hilbert_between)}  # 相邻步之间的距离")

print("\n前 10 步 d_H(w_t, w*):")
print([float(x) for x in hilbert_to_final[:10]])

print("\n前 10 步 d_H(w_t, w_0):")
print([float(x) for x in hilbert_to_init[:10]])

print("\n前 10 步 d_H(w_t, w_{t-1}):")
print([float(x) for x in hilbert_between[:10]])

# ============================
# 额外：收缩比率（几何收缩性）
# ============================
ratios_to_final = []
for t in range(len(hilbert_to_final) - 1):
    if hilbert_to_final[t] > 0:
        ratios_to_final.append(hilbert_to_final[t+1] / hilbert_to_final[t])
    else:
        ratios_to_final.append(float("nan"))

ratios_between = []
for t in range(len(hilbert_between) - 1):
    if hilbert_between[t] > 0:
        ratios_between.append(hilbert_between[t+1] / hilbert_between[t])
    else:
        ratios_between.append(float("nan"))

print("\n前 20 个 ratio: d_H(w_{t+1}, w*) / d_H(w_t, w*):")
print(ratios_to_final[:20])

# 简单做一点统计：去掉前几步的剧烈抖动
burn_in = 20
if len(ratios_to_final) > burn_in + 10:
    tail = ratios_to_final[burn_in:]
    tail_clean = [r for r in tail if not math.isnan(r) and r != float("inf")]
    if len(tail_clean) > 0:
        print(f"\n从 step>{burn_in} 之后的 ratio_to_final：")
        print(f"  平均值 ≈ {sum(tail_clean)/len(tail_clean):.4f}")
        print(f"  最小值 ≈ {min(tail_clean):.4f}, 最大值 ≈ {max(tail_clean):.4f}")



=== Hilbert 分析结果 (部分) ===
len(hilbert_to_final)  = 1408  # 每个 step 对 w* 的距离
len(hilbert_to_init)   = 1408   # 每个 step 对 w0 的距离
len(hilbert_between)   = 1407  # 相邻步之间的距离

前 10 步 d_H(w_t, w*):
[13.656726837158203, 13.419578552246094, 13.898550033569336, 13.527496337890625, 12.841720581054688, 13.374065399169922, 12.373943328857422, 14.936422348022461, 13.295469284057617, 13.550796508789062]

前 10 步 d_H(w_t, w_0):
[0.0, 2.9013776779174805, 4.150603294372559, 5.575801849365234, 4.823336601257324, 6.833193778991699, 5.641814231872559, 8.132007598876953, 7.649444580078125, 7.533379077911377]

前 10 步 d_H(w_t, w_{t-1}):
[2.9013776779174805, 2.0438406467437744, 3.8183467388153076, 1.483393669128418, 3.898033618927002, 3.74942684173584, 4.775698661804199, 4.3495988845825195, 4.156457424163818, 4.757666110992432]

前 20 个 ratio: d_H(w_{t+1}, w*) / d_H(w_t, w*):
[0.9826350568668578, 1.035691991328824, 0.9733027046143302, 0.9493050495297438, 1.0414543218531478, 0.9252192926786066, 1.207086694278698

In [ ]:
print("\n最后 200 步的 ratio_to_final：")
print(f"  平均值 ≈ {sum(ratios_tail)/len(ratios_tail):.4f}")
print(f"  最小值 ≈ {min(ratios_tail):.4f}, 最大值 ≈ {max(ratios_tail):.4f}")


In [ ]:
if sys.path and "swiss_roll_models" in sys.path[0]:
    sys.path.pop(0)

# 或者打印一下看看路径情况
print(sys.path)
import matplotlib.pyplot as plt
# ============================
# 画图：Hilbert 距离和收缩比率
# ============================
steps = list(range(len(hilbert_to_final)))

plt.figure()
plt.semilogy(steps, hilbert_to_final)
plt.xlabel("step t")
plt.ylabel("d_H(w_t, w*) (log scale)")
plt.title("Hilbert distance to w* during training")
plt.tight_layout()
plt.savefig("hilbert_to_final.png")

# 为了避免太密，下采样一点再画 ratio
stride = 10
ratio_idx = list(range(0, len(ratios_to_final), stride))
ratio_vals = [ratios_to_final[i] for i in ratio_idx]

plt.figure()
plt.plot(ratio_idx, ratio_vals)
plt.xlabel("step t")
plt.ylabel("ratio d_H(w_{t+1}, w*) / d_H(w_t, w*)")
plt.axhline(1.0, linestyle="--")
plt.title(f"Hilbert contraction ratio (stride={stride})")
plt.tight_layout()
plt.savefig("hilbert_ratio_to_final.png")


['c:\\Users\\ASUS\\Desktop\\cone_dynamics', 'c:\\Users\\ASUS\\Desktop\\cone_dynamics', 'c:\\Users\\ASUS\\Desktop\\cone_dynamics', 'c:\\Users\\ASUS\\anaconda3\\envs\\py311\\python311.zip', 'c:\\Users\\ASUS\\anaconda3\\envs\\py311\\DLLs', 'c:\\Users\\ASUS\\anaconda3\\envs\\py311\\Lib', 'c:\\Users\\ASUS\\anaconda3\\envs\\py311', '', 'c:\\Users\\ASUS\\anaconda3\\envs\\py311\\Lib\\site-packages', 'c:\\Users\\ASUS\\anaconda3\\envs\\py311\\Lib\\site-packages\\win32', 'c:\\Users\\ASUS\\anaconda3\\envs\\py311\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\ASUS\\anaconda3\\envs\\py311\\Lib\\site-packages\\Pythonwin']


ModuleNotFoundError: No module named 'matplotlib'